# Example 1: HiggsML Challenge with Logistic Regression

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn import metrics
import matplotlib.pyplot as plt

In [2]:
# Set path to dataset
dataset_file_path = Path('./data/dataset.csv.gz')

In [3]:
# Load the dataset into a pandas DataFrame
df = pd.read_csv(dataset_file_path)

In [4]:
def get_subset(df, subset):
    """Function to extract features, labels, and weights for a given data subset
    """
    df = df[df['KaggleSet'] == subset]
    y_true = (df.pop('Label') == 's').to_numpy(np.int64)  # Convert 's'/'b' labels to 1/0
    weight = df.pop('KaggleWeight').to_numpy(np.float32)  # Extract weights
    x = df.drop(columns=['EventId', 'Weight', 'KaggleSet'])  # Drop non-feature columns
    x_feature_names = list(x.columns)  # Store feature names
    x = x.to_numpy(np.float32)  # Convert to numpy array
    return x, y_true, weight, x_feature_names

In [5]:
# Split data into training and validation sets
x_train, y_true_train, weight_train, x_feature_names = get_subset(df, 't')
x_val, y_true_val, weight_val, _ = get_subset(df, 'b')

# Preprocessing

## Handle missing values (-999) by replacing them with 0

In [ ]:
imputer = SimpleImputer(missing_values=-999, strategy='constant', fill_value=0)
x_train = imputer.fit_transform(x_train)
x_val = imputer.transform(x_val)

## Apply log1p transformation to long-tailed features

In [7]:
long_tail_feature_list = [
    'PRI_tau_pt', 'PRI_lep_pt', 'PRI_met', 'PRI_met_sumet', 'PRI_jet_all_pt',
    'PRI_jet_leading_pt', 'PRI_jet_subleading_pt', 'DER_mass_MMC',
    'DER_mass_transverse_met_lep', 'DER_mass_vis', 'DER_pt_h',
    'DER_mass_jet_jet', 'DER_pt_tot', 'DER_sum_pt', 'DER_pt_ratio_lep_tau',
]
for each in long_tail_feature_list:
    print(each)
    feature_idx = x_feature_names.index(each)
    x_train[:, feature_idx] = np.log1p(x_train[:, feature_idx])
    x_val[:, feature_idx] = np.log1p(x_val[:, feature_idx])

PRI_tau_pt
PRI_lep_pt
PRI_met
PRI_met_sumet
PRI_jet_all_pt
PRI_jet_leading_pt
PRI_jet_subleading_pt
DER_mass_MMC
DER_mass_transverse_met_lep
DER_mass_vis
DER_pt_h
DER_mass_jet_jet
DER_pt_tot
DER_sum_pt
DER_pt_ratio_lep_tau


## Standardize features (zero mean, unit variance)

In [8]:
x_scaler = StandardScaler()
x_train = x_scaler.fit_transform(x_train)
x_val = x_scaler.transform(x_val)

In [9]:
# Model Training

In [10]:
# Use SGDClassifier with logistic loss (logistic regression)
model = SGDClassifier(loss='log_loss', max_iter=100)
model.fit(X=x_train, y=y_true_train)

SGDClassifier(loss='log_loss', max_iter=100)

# Evaluation

## Predict probabilities for validation set

In [ ]:
y_score_val = model.predict_proba(x_val)[:, 1] # Keep probability of class 1 (signal)

## Score Distributions

### Plot normalized histograms of prediction scores for signal and background

In [ ]:
fig, ax = plt.subplots()
ax.hist(
    x=[y_score_val[y_true_val == 0], y_score_val[y_true_val == 1]],
    weights=[weight_val[y_true_val == 0], weight_val[y_true_val == 1]],
    label=['Background', 'Signal'],
    color=['tab:orange', 'tab:blue'],
    histtype='step',
    density=True,
    lw=2,
    bins=40,
)
ax.set_xlabel('Score')
ax.set_ylabel('a.u.')
ax.legend()

### Plot stacked bar histogram on log scale

In [ ]:
fig, ax = plt.subplots()
ax.hist(
    x=[y_score_val[y_true_val == 1], y_score_val[y_true_val == 0]],
    weights=[weight_val[y_true_val == 1], weight_val[y_true_val == 0]],
    label=['Signal', 'Background'],
    color=['tab:blue', 'tab:orange'],
    histtype='barstacked',
    lw=2,
    bins=40,
)
ax.set_xlabel('Score')
ax.set_ylabel('Events')
ax.set_yscale('log')

### Plot stacked bar histogram without log scale

In [ ]:
fig, ax = plt.subplots()
ax.hist(
    x=[y_score_val[y_true_val == 0], y_score_val[y_true_val == 1]],
    weights=[weight_val[y_true_val == 0], weight_val[y_true_val == 1]],
    label=['Background', 'Signal'],
    color=['tab:orange', 'tab:blue'],
    histtype='barstacked',
    lw=2,
    bins=40,
)
ax.set_xlabel('Score')
ax.set_ylabel('Events')

## ROC Curve
- fpr: false positive rate
- tpr: true positive rate = signal efficiency
- tnr: true negative rate = background rejection rate

In [20]:
fpr, tpr, _ = metrics.roc_curve(y_true=y_true_val, y_score=y_score_val, sample_weight=weight_val)
tnr = 1 - fpr
auc = metrics.auc(x=tpr, y=tnr)  # AUC for signal efficiency vs. background rejection

Plot ROC curve

In [ ]:
fig, ax = plt.subplots()
ax.plot(tpr, tnr, lw=2)
ax.plot([0, 1], [1, 0], color='gray', lw=2, ls=':', label=f'ROC AUC: {auc:.3f}')
ax.set_xlabel('Signal Efficiency')
ax.set_ylabel('Background Rejection')
ax.legend()
ax.grid()

### 📈 What is AMS (Approximate Median Significance)?

In high energy physics (HEP), AMS is a statistical metric used to quantify how significantly a **signal** stands out from the **background**. It is especially useful when evaluating classifiers designed to distinguish rare events (like Higgs decays) from common background processes.

---

#### 🔍 Purpose

AMS is designed to:

- Handle **class imbalance** between signal and background.
- Incorporate **event weights** that reflect physics-based expectations.
- Estimate the **discovery significance** of a model’s predictions.

---

#### 📐 Formula

$$
\text{AMS} = \sqrt{2\left[(s + b + b_{\text{reg}})\ln\left(1 + \frac{s}{b + b_{\text{reg}}}\right) - s\right]}
$$

Where:

- $ s $: Sum of **weights of selected signal events**
- $ b $: Sum of **weights of selected background events**
- $ b_{\text{reg}} $: Regularization term (set to 10.0 in the Higgs ML Challenge)

---

#### 🧠 Intuition

- AMS is derived from a **likelihood ratio test** used to determine if an observed signal is statistically significant.
- It approximates the **median discovery significance**: how likely we are to reject the background-only hypothesis if signal is truly present.
- In simple terms, it's a physics-aware version of:

$$
\text{AMS}_\text{approx} \approx \frac{s}{\sqrt{b}}
$$

But with improved behavior when \( s \) is not much smaller than \( b \), or when \( b \to 0 \).

---

#### ✅ Why Not Accuracy or AUC?

- **Accuracy** ignores class imbalance.
- **AUC** ranks predictions, but not in a way that directly reflects discovery potential.
- **AMS** focuses on the **signal-to-background trade-off**, which is what matters in physics analyses.

---

Use AMS when your end goal is **discovery**, not just classification performance.

In [23]:
def ams_score(y_true, y_score, weight, threshold: float = 0.5, br: float = 10.0):
    """Function to compute AMS (approximate median significance)
    """
    y_pred = y_score > threshold
    s = weight[(y_true == 1) & (y_pred == 1)].sum()
    b = weight[(y_true == 0) & (y_pred == 1)].sum()
    return np.sqrt(2 * ((s + b + br) * np.log(1.0 + s / (b + br)) - s))

In [24]:
# Evaluate AMS for multiple thresholds
ams_threshold_arr = np.linspace(0, 1)
ams_arr = np.array([
    ams_score(y_true=y_true_val, y_score=y_score_val, weight=weight_val, threshold=threshold)
    for threshold in ams_threshold_arr
])

In [25]:
# Find best threshold and max AMS
best_idx = np.argmax(ams_arr)
max_ams = ams_arr[best_idx]

In [ ]:
# Plot AMS curve
fig, ax = plt.subplots()
ax.plot(ams_threshold_arr, ams_arr, lw=2)
ax.axvline(ams_threshold_arr[best_idx], color='gray', ls=':')
ax.plot([ams_threshold_arr[best_idx]], [ams_arr[best_idx]], ls='', marker='*', color='red', markersize=20, label=f'Max AMS: {max_ams:.3f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('AMS')
ax.legend()
ax.grid()